In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
# initialize groq model 
model = ChatGroq(
    model ="llama-3.3-70b-versatile",
    temperature=0
)

In [4]:
# define state
class BlogState(TypedDict):
    title : str
    outline : str
    content : str
    evaluate : int

In [5]:
# define node (task-1)

def create_outline(state : BlogState) -> BlogState:
    title =state['title']

    prompt = f'Generate a detailed outline for a blog on the topic - {title}'

    outline = model.invoke(prompt).content
    state['outline'] = outline

    return state

In [14]:
# (task-2)

def create_blog(state : BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content
    state['content'] = content
    return state 

In [18]:
# ( task - 3)
def evaluate_blog(state : BlogState) -> BlogState:
    outline = state['outline']
    content = state['content']

    prompt = f"""
Evaluate the following blog based on the outline.
Return ONLY a single integer from 1 to 10.

Outline:
{outline}

Blog:
{content}
"""

    evaluate = int(model.invoke(prompt).content.strip())
    state['evaluate'] = evaluate 
    return state

In [19]:
# create graph 
graph = StateGraph(BlogState)

graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)
graph.add_node("evaluate_blog", evaluate_blog)

graph.add_edge(START, "create_outline")
graph.add_edge("create_outline","create_blog")
graph.add_edge("create_blog", "evaluate_blog")
graph.add_edge("evaluate_blog", END)

workflow = graph.compile()


In [20]:
# execute
initial_state = {'title': 'Rise of AI'}
final_state = workflow.invoke(initial_state)

In [21]:
print(final_state['outline'])

Here is a detailed outline for a blog on the topic "Rise of AI":

**I. Introduction**

* Brief overview of Artificial Intelligence (AI) and its growing importance in modern society
* Thesis statement: The rise of AI is transforming the way we live, work, and interact with each other, and its impact will only continue to grow in the coming years.

**II. History of AI**

* Early beginnings of AI: Dartmouth Summer Research Project (1956) and the first AI program, Logical Theorist (1956)
* Development of AI in the 1960s and 1970s: rule-based systems, expert systems, and the first AI winter
* Resurgence of AI in the 1980s and 1990s: machine learning, neural networks, and the second AI winter
* Modern AI: deep learning, natural language processing, and the current AI boom

**III. Current State of AI**

* Overview of current AI applications:
	+ Virtual assistants (e.g. Siri, Alexa, Google Assistant)
	+ Image and speech recognition
	+ Natural Language Processing (NLP)
	+ Predictive analytics a

In [22]:
print(final_state['content'])

**The Rise of AI: Transforming the Way We Live, Work, and Interact**

Artificial Intelligence (AI) has become an integral part of modern society, transforming the way we live, work, and interact with each other. From virtual assistants like Siri and Alexa to self-driving cars and personalized medicine, AI is revolutionizing numerous aspects of our lives. As we continue to witness the rapid growth of AI, it is essential to understand its history, current state, benefits, challenges, and future implications. In this blog, we will delve into the world of AI, exploring its evolution, applications, and potential impact on humanity.

**A Brief History of AI**

The concept of Artificial Intelligence dates back to the 1950s, when the Dartmouth Summer Research Project on Artificial Intelligence was established in 1956. This project, led by John McCarthy, Marvin Minsky, Nathaniel Rochester, and Claude Shannon, marked the beginning of AI research. The first AI program, Logical Theorist, was devel

In [23]:
print(final_state['evaluate'])

8
